<a href="https://colab.research.google.com/github/paulchilemya2024-beep/DATA-SCIENCE-CS250-/blob/main/Ch5_Data_Wrangling_Exercise_STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Exercise: Cleaning Festival Food Orders
**Introduction to Data Science with Python, Chapter 5: Data Wrangling**

## Scenario
You have been hired as a data analyst for a New Orleans food festival. The festival collected food orders from five vendors over three days, but the data was entered by different volunteers on different devices, and it's a mess. Before the festival can report which vendors earned the most, **you need to clean the data.**

## What you'll practice
By the end of this exercise, you will be able to:
1. Inspect a dataset to find data quality problems
2. Standardize column names
3. Remove duplicate records
4. Standardize inconsistent text and categories
5. Convert columns to the correct data types
6. Handle missing values
7. Detect and handle invalid values and outliers
8. Create new columns from existing data
9. Validate and save a clean dataset

## Instructions
- Make sure **festival_food_orders_messy.csv** is in the same folder as this notebook.
- Work through each step in order. Write your code in the cell marked `# TODO`.
- Use the **Checkpoint** after each step to confirm your work before moving on.
- Answer the reflection questions at the end.

## Step 1: Load and Inspect the Data
Data wrangling always starts with *looking* at the data. Load the CSV into a DataFrame named `df`, then examine it.

**Your tasks:**
- Import pandas and load the file with `pd.read_csv()`
- Print the shape and display the first 10 rows
- Use `df.info()` to check data types
- Use `df.isna().sum()` to count missing values

**Before moving on, write down at least 5 problems you notice.** (Look closely at the column names, vendor names, prices, dates, and payment types.)

✅ **Checkpoint:** The raw data has **64 rows and 8 columns**.

In [ ]:
import pandas as pd

df = pd.read_csv('festival_food_orders_messy.csv')

print('DataFrame Shape:', df.shape)
display(df.head(10))

df.info()
print('\nMissing Values:')
print(df.isna().sum())

DataFrame Shape: (64, 8)


,Order ID,Vendor Name,ITEM,order_date,Qty,Unit Price,Payment Type,Customer Rating (1-5)
0,1009,sweet potato cafe,Sweet Potato Pie,Oct 9 2026,2.0,6.00,Mobile Pay,NaN
1,1006,Thai Nola,Gumbo Ramen,2026-10-11,2.0,14.00,mobile pay,5.0
2,1012,PJs Signature Oysters,Chargrilled Oysters,10/11/2026,1.0,18.00,CASH,4.0
3,1057,orleans brothers,Crawfish Pie,2026-10-10,2.0,$7.00,card,3.0
4,1029,Orleans Brothers,Crawfish Pie,2026-10-11,4.0,7.00,cash,4.0
5,1004,sweet potato cafe,Fish Plate,2026-10-09,NaN,16.00,card,3.0
6,1056,NaN,Vegan Yakamein,10/11/2026,2.0,$13.00,Card,4.0
7,1007,PJs Signature Oysters,Chargrilled Oysters,Oct 11 2026,3.0,18.00,CASH,3.0
8,1040,Sweet Potato Café,Sweet Potato Pie,10/11/2026,4.0,6.00,Mobile Pay,5.0
9,1039,PJs Signature Oysters,Chargrilled Oysters,Oct 9 2026,1.0,$18.00,Credit Card,4.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0    Order ID              64 non-null     int64  
 1   Vendor Name            63 non-null     object 
 2   ITEM                   64 non-null     object 
 3   order_date             64 non-null     object 
 4   Qty                    61 non-null     float64
 5   Unit Price             64 non-null     object 
 6   Payment Type           62 non-null     object 
 7   Customer Rating (1-5)  60 non-null     float64
dtypes: float64(2), int64(1), object(5)
memory usage: 4.1+ KB

Missing Values:
 Order ID                0
Vendor Name              1
ITEM                     0
order_date               0
Qty                      3
Unit Price               0
Payment Type             2
Customer Rating (1-5)    4
dtype: int64


## Step 2: Clean the Column Names
Column names like `" Order ID"` and `"Customer Rating (1-5)"` have extra spaces, capital letters, and special characters that make them hard to use in code.

**Your task:** Rename every column so it is lowercase, has no leading or trailing spaces, and uses underscores instead of spaces. Rename the rating column to simply `rating`.

💡 **Hint:** You can chain string methods on `df.columns`, such as `.str.strip()`, `.str.lower()`, and `.str.replace()`. You can also use `df.rename(columns={...})`.

✅ **Checkpoint:** Your columns should be:
`['order_id', 'vendor_name', 'item', 'order_date', 'qty', 'unit_price', 'payment_type', 'rating']`

In [ ]:
# TODO: Clean the column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.rename(columns={'customer_rating_(1-5)': 'rating'}, inplace=True)
df

,order_id,vendor_name,item,order_date,qty,unit_price,payment_type,rating
0,1009,sweet potato cafe,Sweet Potato Pie,Oct 9 2026,2.0,6.00,Mobile Pay,NaN
1,1006,Thai Nola,Gumbo Ramen,2026-10-11,2.0,14.00,mobile pay,5.0
2,1012,PJs Signature Oysters,Chargrilled Oysters,10/11/2026,1.0,18.00,CASH,4.0
3,1057,orleans brothers,Crawfish Pie,2026-10-10,2.0,$7.00,card,3.0
4,1029,Orleans Brothers,Crawfish Pie,2026-10-11,4.0,7.00,cash,4.0
...,...,...,...,...,...,...,...,...
59,1018,sweet potato cafe,Sweet Potato Pie,Oct 10 2026,NaN,6.00,cash,4.0
60,1031,Froot Orleans,Vegan Yakamein,2026-10-10,2.0,$0.00,Cash,3.0
61,1006,Thai Nola,Gumbo Ramen,2026-10-11,2.0,14.00,mobile pay,5.0
62,1025,THAI NOLA,Spring Rolls,10/11/2026,2.0,8.00,mobile pay,3.0


## Step 3: Remove Duplicate Records
Some orders were accidentally entered more than once. Duplicates would make vendors look like they earned more than they really did.

**Your task:** Count the duplicate rows, then remove them. Confirm that no `order_id` appears twice.

💡 **Hint:** `df.duplicated().sum()` and `df.drop_duplicates()`

✅ **Checkpoint:** You should find **4 duplicate rows**, leaving **60 rows**.

In [ ]:
# TODO: Find and remove duplicate rows
print('Duplicate Rows:', df.duplicated().sum())
df.drop_duplicates(inplace=True)
print('Duplicate Rows:', df.duplicated().sum())

Duplicate Rows: 4
Duplicate Rows: 0


## Step 4: Standardize Text and Categories
Volunteers typed vendor names and payment types differently. To pandas, `"Thai NOLA"`, `"thai nola"`, and `"THAI NOLA "` are three *different* vendors.

**Your tasks:**
1. Strip extra spaces from `vendor_name`, then use `value_counts()` to see every variation.
2. Standardize each vendor to one official name: **Thai NOLA, Orleans Brothers, PJ's Signature Oysters, Froot Orleans, Sweet Potato Cafe**.
3. Standardize `payment_type` so the only categories are **Cash, Card, and Mobile Pay** (treat "Credit Card" as Card).

💡 **Hint:** Convert names to lowercase first, then use a dictionary with `.map()` or `.replace()` to assign the official names. For payment types, try `.str.title()`.

✅ **Checkpoint:** `value_counts()` should show exactly **5 vendors** (plus 1 missing) and **3 payment types** (plus 2 missing).

In [ ]:
# Standardize vendor names
df['vendor_name'] = df['vendor_name'].str.strip()
vendor_mapping = {
    'sweet potato cafe': 'Sweet Potato Cafe',
    'thai nola': 'Thai NOLA',
    'orleans brothers': 'Orleans Brothers',
    "pjs signature oysters": "PJ's Signature Oysters",
    "pj's signature oysters": "PJ's Signature Oysters",
    'froot orleans': 'Froot Orleans'
}
df['vendor_name'] = df['vendor_name'].str.lower().replace(vendor_mapping)
print('Vendor Name Value Counts:')
print(df['vendor_name'].value_counts(dropna=False))

# Standardize payment types
df['payment_type'] = df['payment_type'].str.strip().str.title()
payment_mapping = {
    'Mobile Pay': 'Mobile Pay',
    'Mobilepay': 'Mobile Pay',
    'Cash': 'Cash',
    'Credit Card': 'Card',
    'Card': 'Card'
}
df['payment_type'] = df['payment_type'].replace(payment_mapping)
print('\nPayment Type Value Counts:')
print(df['payment_type'].value_counts(dropna=False))
df

Vendor Name Value Counts:
vendor_name
Orleans Brothers          14
Thai NOLA                 13
PJ's Signature Oysters    12
Sweet Potato Cafe         11
Froot Orleans              9
NaN                        1
Name: count, dtype: int64

Payment Type Value Counts:
payment_type
Card          28
Cash          20
Mobile Pay    10
NaN            2
Name: count, dtype: int64


,order_id,vendor_name,item,order_date,qty,unit_price,payment_type,rating
0,1009,Sweet Potato Cafe,Sweet Potato Pie,Oct 9 2026,2.0,6.00,Mobile Pay,NaN
1,1006,Thai NOLA,Gumbo Ramen,2026-10-11,2.0,14.00,Mobile Pay,5.0
2,1012,PJ's Signature Oysters,Chargrilled Oysters,10/11/2026,1.0,18.00,Cash,4.0
3,1057,Orleans Brothers,Crawfish Pie,2026-10-10,2.0,$7.00,Card,3.0
4,1029,Orleans Brothers,Crawfish Pie,2026-10-11,4.0,7.00,Cash,4.0
5,1004,Sweet Potato Cafe,Fish Plate,2026-10-09,NaN,16.00,Card,3.0
6,1056,NaN,Vegan Yakamein,10/11/2026,2.0,$13.00,Card,4.0
7,1007,PJ's Signature Oysters,Chargrilled Oysters,Oct 11 2026,3.0,18.00,Cash,3.0
8,1040,Sweet Potato Cafe,Sweet Potato Pie,10/11/2026,4.0,6.00,Mobile Pay,5.0
9,1039,PJ's Signature Oysters,Chargrilled Oysters,Oct 9 2026,1.0,$18.00,Card,4.0


## Step 5: Fix Data Types
Look back at your `df.info()` output. The `unit_price` column is stored as text because some prices include a `$` sign, and `order_date` is text in three different formats (`2026-10-09`, `10/09/2026`, and `Oct 9 2026`).

**Your tasks:**
- Remove the `$` from `unit_price` and convert it to a float
- Make sure `qty` is numeric
- Convert `order_date` to a datetime

💡 **Hint:** `.str.replace("$", "", regex=False)`, `.astype(float)`, `pd.to_numeric()`, and `pd.to_datetime(..., format="mixed")`

⚠️ `format="mixed"` requires pandas 2.0 or newer. If it gives an error, try `pd.to_datetime(df["order_date"])` without it.

✅ **Checkpoint:** `df.dtypes` should show `unit_price` as **float64** and `order_date` as **datetime64**.

In [ ]:
# Remove '$' from unit_price and convert to float
df['unit_price'] = df['unit_price'].astype(str).str.replace('$', '', regex=False).astype(float)

# Convert order_date to datetime
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed')

print('Data Types after conversion:')
df.info()

Data Types after conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      55 non-null     int64         
 1   vendor_name   55 non-null     object        
 2   item          55 non-null     object        
 3   order_date    55 non-null     datetime64[ns]
 4   qty           55 non-null     int64         
 5   unit_price    55 non-null     float64       
 6   payment_type  55 non-null     object        
 7   rating        49 non-null     float64       
 8   order_total   55 non-null     float64       
 9   day           55 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(2), object(4)
memory usage: 4.4+ KB


## Step 6: Handle Missing Values
There is no single rule for missing data. You have to decide what makes sense for **each column** and be able to explain your choice.

Use these rules for this dataset:

| Column | Problem | What to do | Why |
|---|---|---|---|
| `vendor_name` | 1 missing | **Fill it in** using the `item` (every item is sold by only one vendor) | We can recover the true value |
| `payment_type` | 2 missing | Fill with **"Unknown"** | The order is still valid; we just don't know how it was paid |
| `qty` | 3 missing | **Drop** those rows | We can't calculate revenue without a quantity |
| `rating` | 4 missing | **Leave as missing** | Not every customer leaves a rating; guessing would distort the average |

💡 **Hint:** `fillna()`, `dropna(subset=[...])`. To recover the vendor, build a mapping from item to vendor using rows where the vendor is known.

✅ **Checkpoint:** After this step you should have **57 rows**, and `rating` should be the **only** column with missing values.

In [ ]:
# Fill missing vendor_name using item
item_to_vendor = df.dropna(subset=['vendor_name']).set_index('item')['vendor_name'].to_dict()
df['vendor_name'] = df.apply(lambda row: item_to_vendor.get(row['item'], row['vendor_name']) if pd.isna(row['vendor_name']) else row['vendor_name'], axis=1)

# Fill missing payment_type with 'Unknown'
df['payment_type'] = df['payment_type'].fillna('Unknown')

# Drop rows with missing qty
df.dropna(subset=['qty'], inplace=True)

# Leave rating as missing

print('Missing Values after handling:')
print(df.isna().sum())

Missing Values after handling:
order_id        0
vendor_name     0
item            0
order_date      0
qty             0
unit_price      0
payment_type    0
rating          4
dtype: int64


## Step 7: Handle Invalid Values and Outliers
Missing data isn't the only problem. Some values are present but **impossible** or **suspicious**.

**Your tasks:**
1. **Quantity:** Use `describe()` to look at `qty`. A quantity can't be negative, and one order of 500 is almost certainly a typo. Remove rows where `qty` is 0 or less, or greater than 20. Then convert `qty` to an integer.
2. **Rating:** Ratings must be between 1 and 5. Replace any rating outside that range with a missing value (don't delete the whole order, since the sale itself is valid).
3. **Price:** One order has a price of $0.00. Replace it with the standard (median) price for that item.

💡 **Hint:** Boolean filtering such as `df[(df["qty"] > 0) & (df["qty"] <= 20)]`, `.between(1, 5)`, `.loc[]`, and `groupby("item")["unit_price"].median()`

✅ **Checkpoint:** You should have **55 rows**, all ratings between **1 and 5**, and no prices of $0.

In [ ]:
# Handle invalid quantities
df = df[(df['qty'] > 0) & (df['qty'] <= 20)].copy()
df['qty'] = df['qty'].astype(int)
print('DataFrame shape after cleaning quantity:', df.shape)

# Handle invalid ratings
df.loc[~df['rating'].between(1, 5), 'rating'] = float('nan')
print('Missing Values after handling ratings:')
print(df.isna().sum())

# Fix the $0.00 price
median_price_per_item = df.groupby('item')['unit_price'].median()
df.loc[df['unit_price'] == 0.00, 'unit_price'] = df['item'].map(median_price_per_item)
print('\nDataFrame head after fixing price:')
display(df.head())
print('\nMissing Values after handling invalid values:')
print(df.isna().sum())

DataFrame shape after cleaning quantity: (55, 10)
Missing Values after handling ratings:
order_id        0
vendor_name     0
item            0
order_date      0
qty             0
unit_price      0
payment_type    0
rating          6
order_total     0
day             0
dtype: int64

DataFrame head after fixing price:


,order_id,vendor_name,item,order_date,qty,unit_price,payment_type,rating,order_total,day
0,1001,PJ's Signature Oysters,Chargrilled Oysters,2026-10-10,1,18.0,Card,3.0,18.0,Saturday
1,1002,Thai NOLA,Gumbo Ramen,2026-10-09,4,14.0,Card,3.0,56.0,Friday
2,1003,Sweet Potato Cafe,Fish Plate,2026-10-09,2,16.0,Cash,5.0,32.0,Friday
3,1005,Orleans Brothers,Gumbo,2026-10-11,2,12.0,Card,5.0,24.0,Sunday
4,1006,Thai NOLA,Gumbo Ramen,2026-10-11,2,14.0,Mobile Pay,5.0,28.0,Sunday



Missing Values after handling invalid values:
order_id        0
vendor_name     0
item            0
order_date      0
qty             0
unit_price      0
payment_type    0
rating          6
order_total     0
day             0
dtype: int64


## Step 8: Create New Columns
Now that the data is clean, we can create new information from it.

**Your tasks:**
- Create `order_total` = `qty` × `unit_price`
- Create `day` containing the day of the week for each order
- Sort the data by `order_id` and reset the index

💡 **Hint:** `df["order_date"].dt.day_name()`

✅ **Checkpoint:** The festival ran **Friday, Saturday, and Sunday**.

In [ ]:
# Create order_total and day columns
df['order_total'] = df['qty'] * df['unit_price']
df['day'] = df['order_date'].dt.day_name()

# Sort the data by order_id and reset the index
df.sort_values(by='order_id', inplace=True)
df.reset_index(drop=True, inplace=True)

print('DataFrame head with new columns:')
display(df.head())

DataFrame head with new columns:


,order_id,vendor_name,item,order_date,qty,unit_price,payment_type,rating,order_total,day
0,1001,PJ's Signature Oysters,Chargrilled Oysters,2026-10-10,1,18.0,Card,3.0,18.0,Saturday
1,1002,Thai NOLA,Gumbo Ramen,2026-10-09,4,14.0,Card,3.0,56.0,Friday
2,1003,Sweet Potato Cafe,Fish Plate,2026-10-09,2,16.0,Cash,5.0,32.0,Friday
3,1005,Orleans Brothers,Gumbo,2026-10-11,2,12.0,Card,5.0,24.0,Sunday
4,1006,Thai NOLA,Gumbo Ramen,2026-10-11,2,14.0,Mobile Pay,5.0,28.0,Sunday


## Step 9: Validate, Summarize, and Save
Always double-check your cleaned data before using it.

**Your tasks:**
1. Confirm the final row count, missing values, duplicate IDs, and data types.
2. Create a summary table showing each vendor's **number of orders, total revenue, and average rating**, sorted by revenue.
3. Show total revenue by day.
4. Save your clean data as **festival_food_orders_clean.csv**.

✅ **Checkpoint:** Total revenue across all vendors should be **$1,717.00**.

In [ ]:
# 1. Confirm the final row count, missing values, duplicate IDs, and data types.
print('Final DataFrame shape:', df.shape)
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate order IDs:', df['order_id'].duplicated().sum())
print('\nFinal data types:\n')
df.info()

# 2. Create a summary table showing each vendor's number of orders, total revenue, and average rating
vendor_summary = df.groupby('vendor_name').agg(
    num_orders=('order_id', 'count'),
    total_revenue=('order_total', 'sum'),
    average_rating=('rating', 'mean')
).sort_values(by='total_revenue', ascending=False)
print('\nVendor Summary:\n')
display(vendor_summary)

# 3. Show total revenue by day
day_revenue = df.groupby('day')['order_total'].sum().reindex(['Friday', 'Saturday', 'Sunday'])
print('\nTotal Revenue by Day:\n')
display(day_revenue)

# 4. Save your clean data as festival_food_orders_clean.csv
df.to_csv('festival_food_orders_clean.csv', index=False)
print('\nCleaned data saved to festival_food_orders_clean.csv')
print(f"Total revenue across all vendors: ${df['order_total'].sum():.2f}")

Final DataFrame shape: (55, 10)

Missing values:
 order_id        0
vendor_name     0
item            0
order_date      0
qty             0
unit_price      0
payment_type    0
rating          6
order_total     0
day             0
dtype: int64

Duplicate order IDs: 0

Final data types:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      55 non-null     int64         
 1   vendor_name   55 non-null     object        
 2   item          55 non-null     object        
 3   order_date    55 non-null     datetime64[ns]
 4   qty           55 non-null     int64         
 5   unit_price    55 non-null     float64       
 6   payment_type  55 non-null     object        
 7   rating        49 non-null     float64       
 8   order_total   55 non-null     float64       
 9   day           55 non-null     object        
dtypes: da

,num_orders,total_revenue,average_rating
vendor_name,,,
PJ's Signature Oysters,11,504.0,4.363636
Thai NOLA,13,360.0,4.166667
Froot Orleans,10,316.0,3.777778
Orleans Brothers,12,295.0,3.888889
Sweet Potato Cafe,9,242.0,4.250000



Total Revenue by Day:



,order_total
day,
Friday,691.0
Saturday,333.0
Sunday,693.0



Cleaned data saved to festival_food_orders_clean.csv
Total revenue across all vendors: $1717.00


## Reflection Questions
Answer each question in 2–3 sentences in the cell below.

1. Which vendor earned the most revenue? Which day was the busiest?
2. Why did we **drop** rows with a missing quantity but **keep** rows with a missing rating?
3. What would have happened to the revenue totals if you had skipped Step 3 (removing duplicates) or Step 4 (standardizing vendor names)?
4. We treated a quantity of 500 as an error. How could you confirm whether an outlier like this is a mistake or a real order?
5. Name one data entry practice the festival could use next year to prevent these problems.

*Your answers here:*

1.PJ's Signature Oysters

2.We dropped rows with a missing qty because quantity is a fundamental component for calculating order_total (revenue). Without a quantity, the financial value of an order cannot be determined, rendering that specific transaction unusable for revenue analysis.
We kept rows with a missing rating because not all customers provide ratings, and guessing or imputing ratings could artificially skew the average rating for vendors. The absence of a rating does not invalidate the sale itself, so the order still contributes to revenue and other analyses.


3.If Step 3 (removing duplicates) had been skipped, the revenue totals would be artificially inflated. Duplicate orders would be counted multiple times, leading to an overestimation of actual earnings for both individual vendors and the festival overall.
If Step 4 (standardizing vendor names) had been skipped, the revenue totals for each vendor would be fragmented and incorrect. For example, 'Thai NOLA', 'thai nola', and 'THAI NOLA' would be treated as separate entities, making it impossible to get an accurate total revenue for the single vendor 'Thai NOLA'. The overall festival revenue might still be correct (assuming no other issues), but the vendor-specific breakdown would be misleading.

4.To confirm whether an outlier like a quantity of 500 is a mistake or a real order, one could: Consult the original source data or data entry logs to see if there's any notation or clarification for such an unusually large order. Additionally, one could interview the data entry personnel or vendor to understand if a bulk order or an unusual circumstance led to that entry. Contextual knowledge of typical order sizes at the festival would also be crucial in determining its plausibility.

5. The festival could implement dropdown menus or standardized selection fields for vendor_name and payment_type in their data entry system. This would prevent variations in spelling, capitalization, and extra spaces, ensuring consistency from the point of data capture. They could also use input validation rules for numerical fields like qty and unit_price to flag values outside expected ranges immediately